In [1]:
# ============================================
# 02_market_insights.ipynb
# Cell 1 — Setup & Project Context
# ============================================

from pathlib import Path
import pandas as pd

# pandas display (clean & readable)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# ------------------------------------------------
# Detect project root (works from VS Code notebooks)
# ------------------------------------------------
def get_project_root() -> Path:
    cwd = Path.cwd()

    # if running from notebooks/, go one level up
    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data"

print("📍 Current working directory:", Path.cwd())
print("📁 Project root:", PROJECT_ROOT)
print("📂 Data directory exists:", DATA_DIR.exists())

print("\n📦 Project root contents:")
for p in sorted(PROJECT_ROOT.iterdir()):
    print(" -", p.name)


📍 Current working directory: /Users/yuvalelbazberger/Documents/MarketPulse/notebooks
📁 Project root: /Users/yuvalelbazberger/Documents/MarketPulse
📂 Data directory exists: True

📦 Project root contents:
 - .ipynb_checkpoints
 - .venv
 - .virtual_documents
 - Untitled.ipynb
 - data
 - notebooks
 - requirements.txt
 - src


In [2]:
# ============================================
# Cell 2 — Load Market Data
# ============================================

import duckdb
import pandas as pd
from pathlib import Path

# ---------------------------------------------
# Candidate locations for data
# ---------------------------------------------
SEARCH_DIRS = [
    DATA_DIR / "db",
    DATA_DIR / "raw",
]

def load_market_data(search_dirs) -> pd.DataFrame:
    """
    Load market data from DuckDB / Parquet / CSV.
    Priority: DuckDB > Parquet > CSV
    """
    # 1) DuckDB
    for d in search_dirs:
        if d.exists():
            duckdb_files = sorted(d.glob("*.duckdb"))
            if duckdb_files:
                db_path = duckdb_files[0]
                con = duckdb.connect(str(db_path), read_only=True)

                tables = [t[0] for t in con.execute("SHOW TABLES").fetchall()]
                if not tables:
                    raise ValueError(f"DuckDB found ({db_path.name}) but has no tables.")

                preferred = ["prices", "market_data", "data", "ohlc"]
                table = next((t for t in preferred if t in tables), tables[0])

                data = con.execute(f"SELECT * FROM {table}").df()
                con.close()

                print(f"✅ Loaded DuckDB: {db_path.name} | table: {table}")
                return data

    # 2) Parquet
    for d in search_dirs:
        if d.exists():
            parquet_files = sorted(d.glob("*.parquet"))
            if parquet_files:
                p = parquet_files[0]
                print(f"✅ Loaded Parquet: {p.name}")
                return pd.read_parquet(p)

    # 3) CSV
    for d in search_dirs:
        if d.exists():
            csv_files = sorted(d.glob("*.csv"))
            if csv_files:
                c = csv_files[0]
                print(f"✅ Loaded CSV: {c.name}")
                return pd.read_csv(c)

    raise FileNotFoundError(
        "No data found. Expected .duckdb / .parquet / .csv in data/db or data/raw."
    )

# ---------------------------------------------
# Load
# ---------------------------------------------
data = load_market_data(SEARCH_DIRS)

# basic column hygiene
data.columns = [c.strip().lower() for c in data.columns]

print("✅ Data shape:", data.shape)
print("✅ Columns:", list(data.columns))
data.head()


✅ Loaded DuckDB: marketpulse.duckdb | table: features_daily
✅ Data shape: (2259, 6)
✅ Columns: ['date', 'ticker', 'close', 'ma_20', 'ma_50', 'vol_20']


,date,ticker,close,ma_20,ma_50,vol_20
0,2025-01-06,MSFT,427.850006,427.850006,427.850006,NaN
1,2025-01-07,MSFT,422.369995,425.110001,425.110001,NaN
2,2025-01-08,MSFT,424.559998,424.926666,424.926666,0.012723
3,2025-01-10,MSFT,418.950012,423.432503,423.432503,0.010507
4,2025-01-13,MSFT,417.190002,422.184003,422.184003,0.008688


In [3]:
# ============================================
# Cell 3 — Standardize + Daily Return (clean)
# ============================================

# detect key columns (already matches your table, but keep it robust)
date_candidates = ["date", "datetime", "timestamp", "time"]
symbol_candidates = ["ticker", "symbol"]
close_candidates = ["close", "adj_close", "adjclose"]

date_col = next((c for c in date_candidates if c in data.columns), None)
symbol_col = next((c for c in symbol_candidates if c in data.columns), None)
close_col = next((c for c in close_candidates if c in data.columns), None)

if not all([date_col, symbol_col, close_col]):
    raise ValueError(f"Missing required columns. Found: date={date_col}, symbol={symbol_col}, close={close_col}")

# standardize types + sort
data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
data = (
    data.dropna(subset=[date_col, close_col, symbol_col])
        .sort_values([symbol_col, date_col])
        .reset_index(drop=True)
)

# compute daily return per ticker (no loops)
data["daily_return"] = (
    data.groupby(symbol_col)[close_col]
        .pct_change()
)

# a clean view (identity + price + trend + risk + return)
cols_pretty = [date_col, symbol_col, close_col]
for c in ["ma_20", "ma_50", "vol_20", "daily_return"]:
    if c in data.columns:
        cols_pretty.append(c)

data[cols_pretty].head(10)


,date,ticker,close,ma_20,ma_50,vol_20,daily_return
0,2025-01-06,AAPL,245.000000,245.000000,245.000000,NaN,NaN
1,2025-01-07,AAPL,242.210007,243.605003,243.605003,NaN,-0.011388
2,2025-01-08,AAPL,242.699997,243.303335,243.303335,0.009483,0.002023
3,2025-01-10,AAPL,236.850006,241.690002,241.690002,0.013065,-0.024104
4,2025-01-13,AAPL,234.399994,240.232001,240.232001,0.010675,-0.010344
5,2025-01-14,AAPL,233.279999,239.073334,239.073334,0.009649,-0.004778
6,2025-01-15,AAPL,237.869995,238.901428,238.901428,0.014781,0.019676
7,2025-01-16,AAPL,228.259995,237.571249,237.571249,0.019051,-0.040400
8,2025-01-17,AAPL,229.979996,236.727776,236.727776,0.018684,0.007535
9,2025-01-21,AAPL,222.639999,235.318999,235.318999,0.019248,-0.031916


In [4]:
data.groupby("ticker")["date"].max().sort_values(ascending=False).head()


ticker
AAPL    2026-01-06
AMZN    2026-01-06
DIA     2026-01-06
GOOGL   2026-01-06
IWM     2026-01-06
Name: date, dtype: datetime64[us]

In [7]:
# ============================================
# Cell 4 — Insights Snapshot (simple & useful)
# ============================================

# latest date available in the dataset
latest_date = data[date_col].max()

latest = (
    data[data[date_col] == latest_date]
    .loc[:, [date_col, symbol_col, close_col, "daily_return", "ma_20", "ma_50", "vol_20"]]
    .copy()
)

# -----------------------------
# 1) Top movers (daily)
# -----------------------------
top_gainers = (
    latest.sort_values("daily_return", ascending=False)
          .head(10)
          .reset_index(drop=True)
)

top_losers = (
    latest.sort_values("daily_return", ascending=True)
          .head(10)
          .reset_index(drop=True)
)

# -----------------------------
# 2) Trend strength (above MAs)
# -----------------------------
latest["above_ma20"] = latest[close_col] > latest["ma_20"]
latest["above_ma50"] = latest[close_col] > latest["ma_50"]

trend_leaders = (
    latest.assign(trend_score=latest["above_ma20"].astype(int) + latest["above_ma50"].astype(int))
          .sort_values(["trend_score", "daily_return"], ascending=[False, False])
          .head(10)
          .loc[:, [date_col, symbol_col, close_col, "daily_return", "ma_20", "ma_50", "trend_score"]]
          .reset_index(drop=True)
)

# -----------------------------
# 3) Risk (volatility leaders)
# -----------------------------
most_volatile = (
    latest.sort_values("vol_20", ascending=False)
          .head(10)
          .loc[:, [date_col, symbol_col, close_col, "vol_20", "daily_return", "ma_20", "ma_50"]]
          .reset_index(drop=True)
)

print("📅 Latest date in data:", latest_date)

print("\n🚀 Top 10 Gainers (daily % move):")
top_gainers

print("\n🧊 Top 10 Losers (daily % move):")
top_losers

print("\n📈 Trend leaders (above MA20/MA50):")
trend_leaders

print("\n🌪️ Most volatile (last 20 days):")
most_volatile



📅 Latest date in data: 2026-01-06 00:00:00

🚀 Top 10 Gainers (daily % move):

🧊 Top 10 Losers (daily % move):

📈 Trend leaders (above MA20/MA50):

🌪️ Most volatile (last 20 days):


,date,ticker,close,vol_20,daily_return,ma_20,ma_50
0,2026-01-06,NVDA,189.139999,0.018752,0.005422,183.537000,186.807601
1,2026-01-06,AMZN,238.536407,0.013998,0.023498,229.126321,232.177329
2,2026-01-06,GOOGL,312.800507,0.013808,-0.011814,311.524525,300.384611
3,2026-01-06,MSFT,471.410004,0.010631,-0.003045,482.186502,493.674002
4,2026-01-06,IWM,252.699997,0.009190,-0.000119,251.267999,246.397800
5,2026-01-06,QQQ,620.070007,0.008477,0.003366,617.996497,616.846998
6,2026-01-06,AAPL,262.792389,0.006923,-0.016716,273.344118,273.067448
7,2026-01-06,DIA,491.840088,0.006192,0.004227,483.676006,476.974402
8,2026-01-06,SPY,688.830017,0.005573,0.001614,684.160498,679.476002


In [8]:
# Visual ranking helper (still table-only)

view = (
    latest
    .assign(
        daily_return_pct=lambda x: (x["daily_return"] * 100).round(2),
        trend=lambda x: (x[close_col] > x["ma_20"]) & (x[close_col] > x["ma_50"])
    )
    .sort_values("daily_return", ascending=False)
    .loc[:, [symbol_col, "daily_return_pct", "trend", "vol_20"]]
    .reset_index(drop=True)
)

view


,ticker,daily_return_pct,trend,vol_20
0,AMZN,2.35,True,0.013998
1,NVDA,0.54,True,0.018752
2,DIA,0.42,True,0.006192
3,QQQ,0.34,True,0.008477
4,SPY,0.16,True,0.005573
5,IWM,-0.01,True,0.009190
6,MSFT,-0.30,False,0.010631
7,GOOGL,-1.18,True,0.013808
8,AAPL,-1.67,False,0.006923


## 📈 How to Interpret Daily Gains vs. Trend

When analyzing market movements, it is important to distinguish between **short-term price changes** and the **broader trend**.  
A positive daily return does **not necessarily** mean that a stock is in an uptrend — and this is completely normal.

### 🔹 Key Concepts

- **Daily Return (`daily_return`)**  
  Measures the percentage price change from the previous trading day.  
  It answers the question:  
  > *“Did the stock go up or down today?”*

- **Trend (Moving Averages: MA20 / MA50)**  
  Indicates the medium-term direction of the stock based on recent history.  
  It answers the question:  
  > *“Is the stock generally trending upward?”*

These two metrics operate on **different time horizons**, so they can — and often do — disagree.

---

## 🧠 Common Scenarios

### 🟡 Scenario 1: Counter-Trend Bounce (Very Common)
- The stock is below its moving averages (downtrend).
- A positive daily return occurs.

**Interpretation:**  
A short-term technical rebound within a broader downtrend.  
Often referred to as a *technical correction* or *dead cat bounce*.

- `daily_return` → Positive  
- `trend` → False

---

### 🟢 Scenario 2: Early Trend Reversal
- The stock has recently been weak.
- It starts posting positive daily returns.
- It has not yet crossed above key moving averages.

**Interpretation:**  
Potential early signs of a trend change.  
Requires confirmation over the next few days.

- `daily_return` → Positive  
- `trend` → False (for now)

This scenario is often **interesting**, but not yet confirmed.

---

### 🔴 Scenario 3: High-Noise Move
- A small positive daily return.
- High volatility.
- No clear directional structure.

**Interpretation:**  
Short-term noise rather than meaningful momentum.  
Typically not actionable on its own.

- `daily_return` → Positive  
- `trend` → False  
- `volatility` → High

---

### 🟢🟢 Scenario 4: Trend-Confirmed Gain (Ideal Case)
- Positive daily return.
- Price is above MA20 and MA50.

**Interpretation:**  
The stock is moving upward **with trend confirmation**.  
This represents the strongest and most reliable signal.

- `daily_return` → Positive  
- `trend` → True

---

## ✅ Why This Distinction Matters

Relying solely on daily price changes can lead to:
- Overreacting to noise
- Chasing short-term moves
- Ignoring broader market structure

Combining **daily performance** with **trend context** allows for:
- Better signal quality
- Clearer risk assessment
- More professional, explainable insights

This framework mirrors how market insights are typically presented in real-world analyst reports.


In [9]:
# ============================================
# Cell 5 — Final Market Snapshot (Daily)
# ============================================

# latest date in dataset
latest_date = data[date_col].max()

snapshot = (
    data[data[date_col] == latest_date]
    .copy()
)

# position relative to moving averages
snapshot["above_ma20"] = snapshot[close_col] > snapshot["ma_20"]
snapshot["above_ma50"] = snapshot[close_col] > snapshot["ma_50"]

# simplified MA position (human-readable)
def ma_position(row):
    if row["above_ma20"] and row["above_ma50"]:
        return "Above MA20 & MA50"
    if row["above_ma20"] and not row["above_ma50"]:
        return "Above MA20 only"
    if not row["above_ma20"] and row["above_ma50"]:
        return "Above MA50 only"
    return "Below MA20 & MA50"

snapshot["ma_position"] = snapshot.apply(ma_position, axis=1)

# clean, business-ready view
final_snapshot = (
    snapshot
    .assign(
        daily_return_pct=lambda x: (x["daily_return"] * 100).round(2),
        volatility_20d=lambda x: (x["vol_20"] * 100).round(2)
    )
    .loc[:, [
        symbol_col,
        close_col,
        "daily_return_pct",
        "ma_position",
        "volatility_20d"
    ]]
    .sort_values("daily_return_pct", ascending=False)
    .reset_index(drop=True)
)

print(f"📅 Market Snapshot — {latest_date.date()}")
final_snapshot


📅 Market Snapshot — 2026-01-06


,ticker,close,daily_return_pct,ma_position,volatility_20d
0,AMZN,238.536407,2.35,Above MA20 & MA50,1.40
1,NVDA,189.139999,0.54,Above MA20 & MA50,1.88
2,DIA,491.840088,0.42,Above MA20 & MA50,0.62
3,QQQ,620.070007,0.34,Above MA20 & MA50,0.85
4,SPY,688.830017,0.16,Above MA20 & MA50,0.56
5,IWM,252.699997,-0.01,Above MA20 & MA50,0.92
6,MSFT,471.410004,-0.30,Below MA20 & MA50,1.06
7,GOOGL,312.800507,-1.18,Above MA20 & MA50,1.38
8,AAPL,262.792389,-1.67,Below MA20 & MA50,0.69


## 🗞️ Daily Market Summary

This snapshot provides a high-level view of the market based on the latest available trading day.

### 🔹 Top Performers
The strongest daily gains were led by assets with the highest positive daily returns.  
These moves highlight short-term momentum but should always be evaluated in the context of trend and volatility.

### 🔹 Trend Context
Assets trading **above both MA20 and MA50** indicate sustained medium-term strength.  
Gains occurring below these levels may represent short-term rebounds rather than confirmed trends.

### 🔹 Risk & Volatility
Assets with elevated 20-day volatility exhibit larger price swings and higher uncertainty.  
While they may offer upside potential, they also carry increased risk and require cautious interpretation.

### 🔹 Key Takeaway
Daily performance alone does not provide a complete picture.  
Combining **price movement**, **trend positioning**, and **volatility** allows for more balanced and professional market insights.



In [10]:
# ============================================
# Cell 6 — Export Snapshot for Dashboard
# ============================================

from pathlib import Path

EXPORT_DIR = PROJECT_ROOT / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

export_path = EXPORT_DIR / f"market_snapshot_{latest_date.date()}.csv"

final_snapshot.to_csv(export_path, index=False)

print("✅ Snapshot exported to:")
print(export_path)


✅ Snapshot exported to:
/Users/yuvalelbazberger/Documents/MarketPulse/exports/market_snapshot_2026-01-06.csv
